In [26]:


import pandas as pd
import numpy as np

# Create 30 days of simulated sales data
np.random.seed(42)

dates = pd.date_range(
    start="2026-10-01",
    periods=30,
    freq="D"
)

products = {
    "SKU001": 10,
    "SKU002": 15,
    "SKU003": 8
}

sales_records = []

for date in dates:
    for sku, average_sales in products.items():
        sales_records.append({
            "date": date,
            "sku": sku,
            "qty_sold": np.random.poisson(average_sales)
        })

sales = pd.DataFrame(sales_records)

sales.to_csv("sales.csv", index=False)

print("Sales records:", len(sales))
print(sales.head())


Sales records: 90
        date     sku  qty_sold
0 2026-10-01  SKU001        12
1 2026-10-01  SKU002        10
2 2026-10-01  SKU003         7
3 2026-10-02  SKU001         7
4 2026-10-02  SKU002        13


In [27]:

inventory = pd.DataFrame({
    "sku": ["SKU001", "SKU002", "SKU003"],
    "product_name": ["T-Shirt", "Jeans", "Hoodie"],
    "current_stock": [50, 35, 20]
})

inventory.to_csv("inventory.csv", index=False)

print(inventory)


      sku product_name  current_stock
0  SKU001      T-Shirt             50
1  SKU002        Jeans             35
2  SKU003       Hoodie             20


In [28]:


# Create parameters data
params = pd.DataFrame({
    "sku": ["SKU001", "SKU002", "SKU003"],
    "unit_cost": [15.00, 30.00, 25.00],
    "holding_cost_per_day": [0.10, 0.20, 0.15],
    "stockout_cost": [20.00, 40.00, 35.00],
    "lead_time_days": [3, 5, 7],
    "min_order_qty": [10, 15, 10],
    "service_level": [0.95, 0.95, 0.95]
})

# Save parameters file
params.to_csv("params.csv", index=False)

print("Parameters Data:")
print(params.to_string(index=False))


Parameters Data:
   sku  unit_cost  holding_cost_per_day  stockout_cost  lead_time_days  min_order_qty  service_level
SKU001       15.0                  0.10           20.0               3             10           0.95
SKU002       30.0                  0.20           40.0               5             15           0.95
SKU003       25.0                  0.15           35.0               7             10           0.95


In [29]:

# Load the three CSV files
sales_data = pd.read_csv("sales.csv")
inventory_data = pd.read_csv("inventory.csv")
params_data = pd.read_csv("params.csv")

# Check that the files loaded successfully
print("Sales Data:")
print(sales_data)

print("\nInventory Data:")
print(inventory_data)

print("\nParameters Data:")
print(params_data)


Sales Data:
          date     sku  qty_sold
0   2026-10-01  SKU001        12
1   2026-10-01  SKU002        10
2   2026-10-01  SKU003         7
3   2026-10-02  SKU001         7
4   2026-10-02  SKU002        13
..         ...     ...       ...
85  2026-10-29  SKU002        19
86  2026-10-29  SKU003         4
87  2026-10-30  SKU001        17
88  2026-10-30  SKU002        16
89  2026-10-30  SKU003         8

[90 rows x 3 columns]

Inventory Data:
      sku product_name  current_stock
0  SKU001      T-Shirt             50
1  SKU002        Jeans             35
2  SKU003       Hoodie             20

Parameters Data:
      sku  unit_cost  holding_cost_per_day  stockout_cost  lead_time_days  \
0  SKU001       15.0                  0.10           20.0               3   
1  SKU002       30.0                  0.20           40.0               5   
2  SKU003       25.0                  0.15           35.0               7   

   min_order_qty  service_level  
0             10           0.95  
1    

In [30]:


# Load updated sales data
sales_data = pd.read_csv("sales.csv")

# Check for missing values
print("Missing Values:")
print(sales_data.isnull().sum())

# Check for duplicates
print("\nDuplicate Records:")
print(sales_data.duplicated().sum())

# Remove exact duplicates
sales_data = sales_data.drop_duplicates().copy()

# Convert dates to correct format
sales_data["date"] = pd.to_datetime(
    sales_data["date"], errors="coerce"
)

# Convert quantities to numbers
sales_data["qty_sold"] = pd.to_numeric(
    sales_data["qty_sold"], errors="coerce"
)

# Remove missing or invalid records
sales_data = sales_data.dropna(
    subset=["date", "sku", "qty_sold"]
)

# Remove negative quantities
sales_data = sales_data[
    sales_data["qty_sold"] >= 0
].copy()

print("\nCleaned Sales Records:", len(sales_data))
print(sales_data.head(10))


Missing Values:
date        0
sku         0
qty_sold    0
dtype: int64

Duplicate Records:
0

Cleaned Sales Records: 90
        date     sku  qty_sold
0 2026-10-01  SKU001        12
1 2026-10-01  SKU002        10
2 2026-10-01  SKU003         7
3 2026-10-02  SKU001         7
4 2026-10-02  SKU002        13
5 2026-10-02  SKU003         8
6 2026-10-03  SKU001        10
7 2026-10-03  SKU002        11
8 2026-10-03  SKU003         9
9 2026-10-04  SKU001        15


In [31]:

# Aggregate daily demand by SKU
daily_demand = sales_data.groupby(
    ["date", "sku"], as_index=False
)["qty_sold"].sum()

# Rename column for forecasting
daily_demand = daily_demand.rename(
    columns={"qty_sold": "daily_demand"}
)

# Sort by date and SKU
daily_demand = daily_demand.sort_values(
    by=["date", "sku"]
)

# Save the aggregated data
daily_demand.to_csv("daily_demand.csv", index=False)

print("Daily Demand by SKU:")
print(daily_demand)


Daily Demand by SKU:
         date     sku  daily_demand
0  2026-10-01  SKU001            12
1  2026-10-01  SKU002            10
2  2026-10-01  SKU003             7
3  2026-10-02  SKU001             7
4  2026-10-02  SKU002            13
..        ...     ...           ...
85 2026-10-29  SKU002            19
86 2026-10-29  SKU003             4
87 2026-10-30  SKU001            17
88 2026-10-30  SKU002            16
89 2026-10-30  SKU003             8

[90 rows x 3 columns]


In [32]:

import numpy as np

# Make random results reproducible
np.random.seed(42)

# Create 30 days of dates
dates = pd.date_range(
    start="2026-10-01",
    periods=30,
    freq="D"
)

# Average daily sales for each product
products = {
    "SKU001": 10,
    "SKU002": 15,
    "SKU003": 8
}

# Generate simulated daily sales
sales_records = []

for date in dates:
    for sku, average_sales in products.items():
        units_sold = np.random.poisson(average_sales)

        sales_records.append({
            "date": date,
            "sku": sku,
            "units_sold": units_sold
        })

# Replace the original sample with 30 days of data
sales = pd.DataFrame(sales_records)

# Save the expanded sales dataset
sales.to_csv("sales.csv", index=False)

print("Total Sales Records:", len(sales))
print(sales.head(10))


Total Sales Records: 90
        date     sku  units_sold
0 2026-10-01  SKU001          12
1 2026-10-01  SKU002          10
2 2026-10-01  SKU003           7
3 2026-10-02  SKU001           7
4 2026-10-02  SKU002          13
5 2026-10-02  SKU003           8
6 2026-10-03  SKU001          10
7 2026-10-03  SKU002          11
8 2026-10-03  SKU003           9
9 2026-10-04  SKU001          15


In [33]:

# Check total records
print("Total Records:", len(daily_demand))

# Check number of products
print("Unique Products:", daily_demand["sku"].nunique())

# Check days of data per product
print("\nDays per Product:")
print(daily_demand.groupby("sku")["date"].nunique())

# Check missing values
print("\nMissing Values:")
print(daily_demand.isnull().sum())

# Check duplicate date-SKU combinations
print("\nDuplicate Date-SKU Records:")
print(daily_demand.duplicated(["date", "sku"]).sum())

# Check that all products match across files
sales_skus = set(daily_demand["sku"])
inventory_skus = set(inventory_data["sku"])
params_skus = set(params_data["sku"])

print("\nAll SKUs Match:",
      sales_skus == inventory_skus == params_skus)

# Check for invalid demand
print("Negative Demand:",
      (daily_demand["daily_demand"] < 0).sum())


Total Records: 90
Unique Products: 3

Days per Product:
sku
SKU001    30
SKU002    30
SKU003    30
Name: date, dtype: int64

Missing Values:
date            0
sku             0
daily_demand    0
dtype: int64

Duplicate Date-SKU Records:
0

All SKUs Match: True
Negative Demand: 0


In [34]:

import pandas as pd

# Fix sales.csv
sales = pd.read_csv("sales.csv")
sales = sales.rename(
    columns={"units_sold": "qty_sold"}
)
sales.to_csv("sales.csv", index=False)

# Fix inventory.csv
inventory = pd.read_csv("inventory.csv")
inventory = inventory.rename(
    columns={"current_stock": "opening_stock"}
)
inventory.to_csv("inventory.csv", index=False)

print("Sales Columns:", sales.columns.tolist())
print("Inventory Columns:", inventory.columns.tolist())
print("Both files updated successfully!")



Sales Columns: ['date', 'sku', 'qty_sold']
Inventory Columns: ['sku', 'product_name', 'opening_stock']
Both files updated successfully!


**Demand Forecasting**

In [35]:
# Member 2: prepare demand data and calculate an EWMA forecast
import numpy as np
import pandas as pd

# Reload the corrected sales file so this works after the earlier notebook cells
sales_data = pd.read_csv("sales.csv")
sales_data["date"] = pd.to_datetime(sales_data["date"])
sales_data = sales_data.rename(columns={"units_sold": "qty_sold"})
daily_demand = (sales_data.groupby(["date", "sku"], as_index=False)["qty_sold"]
                .sum().rename(columns={"qty_sold": "actual_demand"})
                .sort_values(["sku", "date"]))

# EWMA smoothing value: larger alpha reacts more quickly to recent changes
ALPHA = 0.30
forecast_rows = []
next_day_forecasts = []

for sku, group in daily_demand.groupby("sku"):
    group = group.reset_index(drop=True)
    ewma_value = float(group.loc[0, "actual_demand"])
    group["predicted_demand"] = np.nan

    # The forecast for today uses the EWMA available before current day sales
    for i in range(1, len(group)):
        group.loc[i, "predicted_demand"] = ewma_value
        ewma_value = (ALPHA * group.loc[i, "actual_demand"]
                      + (1 - ALPHA) * ewma_value)

    forecast_rows.append(group)
    next_day_forecasts.append({"sku": sku, "next_day_forecast": round(ewma_value, 2)})

forecast_results = pd.concat(forecast_rows, ignore_index=True)
next_day_forecast = pd.DataFrame(next_day_forecasts)
forecast_results.to_csv("forecast_results.csv", index=False)
next_day_forecast.to_csv("next_day_forecast.csv", index=False)

print("EWMA Forecast Results:")
display(next_day_forecast)


EWMA Forecast Results:


,sku,next_day_forecast
0,SKU001,13.33
1,SKU002,16.77
2,SKU003,7.32


In [36]:
# Member 2: check forecast accuracy (hanzon)
accuracy_rows = []

for sku, group in forecast_results.groupby("sku"):
    scored = group.dropna(subset=["predicted_demand"]).copy()
    errors = scored["actual_demand"] - scored["predicted_demand"]
    mae = errors.abs().mean()
    rmse = np.sqrt((errors ** 2).mean())
    nonzero_actuals = scored["actual_demand"].replace(0, np.nan)
    mape = (errors.abs() / nonzero_actuals).mean() * 100
    accuracy_rows.append({"sku": sku, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE_percent": round(mape, 2), "accuracy_percent": round(max(0, 100 - mape), 2)})

accuracy_results = pd.DataFrame(accuracy_rows)
accuracy_results.to_csv("forecast_accuracy.csv", index=False)

print("Forecast Accuracy:")
display(accuracy_results)
print("\\nMetric meanings:")
print("MAE = average number of units missed by the forecast.")
print("RMSE = gives extra weight to larger forecast mistakes.")
print("MAPE = average percentage error; lower is better.")
print("Accuracy = 100 - MAPE; higher is better.")


Forecast Accuracy:


,sku,MAE,RMSE,MAPE_percent,accuracy_percent
0,SKU001,3.45,4.04,50.04,49.96
1,SKU002,3.10,3.85,20.07,79.93
2,SKU003,1.91,2.45,27.50,72.50


\nMetric meanings:
MAE = average number of units missed by the forecast.
RMSE = gives extra weight to larger forecast mistakes.
MAPE = average percentage error; lower is better.
Accuracy = 100 - MAPE; higher is better.


In [37]:
# Member 3 (Devin)

import numpy as np
import pandas as pd
from scipy.stats import norm

params_data = pd.read_csv("params.csv")
forecast_results = pd.read_csv("forecast_results.csv")

forecast_results["error"] = (
    forecast_results["actual_demand"] - forecast_results["predicted_demand"]
)

error_std = (
    forecast_results.groupby("sku")["error"]
    .std()
    .reset_index(name="error_std")
)

mean_demand = (
    forecast_results.groupby("sku")["actual_demand"]
    .mean()
    .reset_index(name="mean_daily_demand")
)

policy_df = params_data.merge(error_std, on="sku").merge(
    mean_demand, on="sku"
)

policy_df["z_score"] = policy_df["service_level"].apply(
    lambda sl: norm.ppf(sl)
)

policy_df["safety_stock"] = np.ceil(
    policy_df["z_score"]
    * policy_df["error_std"]
    * np.sqrt(policy_df["lead_time_days"])
)

policy_df["reorder_point"] = np.ceil(
    (policy_df["mean_daily_demand"] * policy_df["lead_time_days"])
    + policy_df["safety_stock"]
)

inventory_policy = policy_df[
    [
        "sku",
        "service_level",
        "lead_time_days",
        "mean_daily_demand",
        "error_std",
        "safety_stock",
        "reorder_point",
    ]
].copy()

inventory_policy.to_csv("inventory_policy.csv", index=False)

print("Inventory Policy (Safety Stock & Reorder Points):")
display(inventory_policy)

Inventory Policy (Safety Stock & Reorder Points):


,sku,service_level,lead_time_days,mean_daily_demand,error_std,safety_stock,reorder_point
0,SKU001,0.95,3,9.566667,4.111703,12.0,41.0
1,SKU002,0.95,5,15.300000,3.838913,15.0,92.0
2,SKU003,0.95,7,8.100000,2.491789,11.0,68.0


In [38]:

# MEMBER 4 – AGENT DECISION LOGIC

import pandas as pd
import numpy as np
import math

inventory = pd.read_csv("inventory.csv")
forecasts = pd.read_csv("next_day_forecast.csv")
policy = pd.read_csv("inventory_policy.csv")
params = pd.read_csv("params.csv")

agent_data = (
    inventory
    .merge(forecasts, on="sku", validate="one_to_one")
    .merge(
        policy[["sku", "safety_stock", "reorder_point"]],
        on="sku",
        validate="one_to_one"
    )
    .merge(
        params[["sku", "lead_time_days", "min_order_qty"]],
        on="sku",
        validate="one_to_one"
    )
)

pending_orders = pd.DataFrame(
    columns=["sku", "quantity", "days_until_arrival"]
)


def make_inventory_decisions(data, pending_orders=None):
    """
    Decide whether to ORDER or WAIT for every SKU.

    Uses:
    - Opening/current stock
    - Forecasted daily demand
    - Safety stock and reorder points
    - Supplier lead times
    - Minimum order quantities
    - Outstanding purchase orders
    """

    if pending_orders is None:
        pending_orders = pd.DataFrame(
            columns=["sku", "quantity", "days_until_arrival"]
        )

    decisions = []

    for _, item in data.iterrows():

        sku = item["sku"]
        stock = float(item["opening_stock"])
        demand = float(item["next_day_forecast"])
        reorder_point = float(item["reorder_point"])
        safety_stock = float(item["safety_stock"])
        lead_time = int(item["lead_time_days"])
        moq = int(item["min_order_qty"])

        if (
            stock < 0 or demand < 0
            or reorder_point < 0 or safety_stock < 0
            or lead_time < 1 or moq < 1
        ):
            raise ValueError(f"Invalid values for {sku}")


        sku_orders = pending_orders[
            pending_orders["sku"] == sku
        ]

        incoming_qty = float(sku_orders["quantity"].sum())

        arriving_soon = float(
            sku_orders.loc[
                sku_orders["days_until_arrival"] <= lead_time,
                "quantity"
            ].sum()
        )


        inventory_position = stock + incoming_qty


        projected_inventory = (
            stock + arriving_soon - demand * lead_time
        )


        target_stock = math.ceil(
            2 * demand * lead_time + safety_stock
        )


        if inventory_position <= reorder_point:

            required_qty = math.ceil(
                target_stock - inventory_position
            )

            order_qty = max(moq, required_qty)
            decision = "ORDER"

            reason = (
                f"Inventory position {inventory_position:.0f} "
                f"is at or below reorder point "
                f"{reorder_point:.0f}. "
                f"Order {order_qty} units."
            )

        else:

            order_qty = 0
            decision = "WAIT"

            reason = (
                f"Inventory position {inventory_position:.0f} "
                f"is above reorder point "
                f"{reorder_point:.0f}. "
                "No order needed."
            )

        decisions.append({
            "sku": sku,
            "current_stock": stock,
            "forecasted_demand": demand,
            "lead_time_days": lead_time,
            "safety_stock": safety_stock,
            "reorder_point": reorder_point,
            "incoming_orders": incoming_qty,
            "inventory_position": inventory_position,
            "projected_inventory": round(
                projected_inventory, 2
            ),
            "decision": decision,
            "order_quantity": order_qty,
            "reason": reason
        })

    return pd.DataFrame(decisions)



agent_results = make_inventory_decisions(
    agent_data, pending_orders
)

agent_results.to_csv(
    "agent_decisions.csv", index=False
)

print("MEMBER 4 – INVENTORY AGENT DECISIONS")

display(agent_results[[
    "sku",
    "current_stock",
    "inventory_position",
    "reorder_point",
    "projected_inventory",
    "decision",
    "order_quantity"
]])

print("\nDECISION EXPLANATIONS")

for _, row in agent_results.iterrows():
    print(f"{row['sku']}: {row['reason']}")

print("\nAgent decisions saved to agent_decisions.csv")


MEMBER 4 – INVENTORY AGENT DECISIONS


,sku,current_stock,inventory_position,reorder_point,projected_inventory,decision,order_quantity
0,SKU001,50.0,50.0,41.0,10.01,WAIT,0
1,SKU002,35.0,35.0,92.0,-48.85,ORDER,148
2,SKU003,20.0,20.0,68.0,-31.24,ORDER,94



DECISION EXPLANATIONS
SKU001: Inventory position 50 is above reorder point 41. No order needed.
SKU002: Inventory position 35 is at or below reorder point 92. Order 148 units.
SKU003: Inventory position 20 is at or below reorder point 68. Order 94 units.

Agent decisions saved to agent_decisions.csv


Simulation & Performance Evaluation

In [39]:

# SIMULATION & PERFORMANCE EVALUATION

import pandas as pd

# Load data from previous members
sales = pd.read_csv("sales.csv")
inventory = pd.read_csv("inventory.csv")
params = pd.read_csv("params.csv")
forecasts = pd.read_csv("next_day_forecast.csv")
policy = pd.read_csv("inventory_policy.csv")

# Prepare simulation data
sales["date"] = pd.to_datetime(sales["date"])

simulation_data = (
    inventory
    .merge(params, on="sku")
    .merge(forecasts, on="sku")
    .merge(
        policy[["sku", "safety_stock", "reorder_point"]],
        on="sku"
    )
)

# Starting inventory for each product
stock = dict(zip(
    simulation_data["sku"],
    simulation_data["opening_stock"]
))

pending_orders = []
simulation_records = []
order_records = []

# Simulate each day
for day, date in enumerate(sorted(sales["date"].unique())):

    for _, item in simulation_data.iterrows():

        sku = item["sku"]

        # 1. Receive purchase orders arriving today
        arrivals = sum(
            order["quantity"]
            for order in pending_orders
            if order["sku"] == sku
            and order["arrival_day"] == day
        )

        stock[sku] += arrivals

        pending_orders = [
            order for order in pending_orders
            if not (
                order["sku"] == sku
                and order["arrival_day"] == day
            )
        ]

        # 2. Get actual customer demand
        demand = int(
            sales.loc[
                (sales["date"] == date) &
                (sales["sku"] == sku),
                "qty_sold"
            ].sum()
        )

        # 3. Fulfill available demand
        units_sold = min(stock[sku], demand)
        stockouts = max(0, demand - units_sold)

        stock[sku] -= units_sold

        # 4. Ask for a decision
        incoming = [
            {
                "sku": order["sku"],
                "quantity": order["quantity"],
                "days_until_arrival":
                    order["arrival_day"] - day
            }
            for order in pending_orders
        ]

        agent_input = pd.DataFrame([{
            "sku": sku,
            "opening_stock": stock[sku],
            "next_day_forecast": item["next_day_forecast"],
            "safety_stock": item["safety_stock"],
            "reorder_point": item["reorder_point"],
            "lead_time_days": item["lead_time_days"],
            "min_order_qty": item["min_order_qty"]
        }])

        pending_df = pd.DataFrame(
            incoming,
            columns=[
                "sku", "quantity", "days_until_arrival"
            ]
        )

        decision = make_inventory_decisions(
            agent_input, pending_df
        ).iloc[0]

        order_qty = int(decision["order_quantity"])

        # 5. Schedule new purchase orders
        if order_qty > 0:
            arrival_day = day + int(item["lead_time_days"])

            pending_orders.append({
                "sku": sku,
                "quantity": order_qty,
                "arrival_day": arrival_day
            })

            order_records.append({
                "date": date,
                "sku": sku,
                "quantity": order_qty,
                "arrival_day": arrival_day
            })

        # 6. Calculate daily costs
        holding_cost = (
            stock[sku] * item["holding_cost_per_day"]
        )

        stockout_cost = (
            stockouts * item["stockout_cost"]
        )

        # Assumption: unit_cost is the purchase cost
        order_cost = order_qty * item["unit_cost"]

        # 7. Save daily simulation results
        simulation_records.append({
            "date": date,
            "sku": sku,
            "demand": demand,
            "units_sold": units_sold,
            "stockouts": stockouts,
            "orders_received": arrivals,
            "ending_stock": stock[sku],
            "decision": decision["decision"],
            "order_quantity": order_qty,
            "holding_cost": holding_cost,
            "stockout_cost": stockout_cost,
            "order_cost": order_cost
        })

# Convert results into DataFrames
simulation_results = pd.DataFrame(simulation_records)
purchase_orders = pd.DataFrame(order_records)

# Save outputs
simulation_results.to_csv(
    "simulation_results.csv", index=False
)

purchase_orders.to_csv(
    "purchase_orders.csv", index=False
)

print("DAILY INVENTORY SIMULATION")
display(simulation_results.head(15))

print("Simulation completed successfully!")


DAILY INVENTORY SIMULATION


,date,sku,demand,units_sold,stockouts,orders_received,ending_stock,decision,order_quantity,holding_cost,stockout_cost,order_cost
0,2026-10-01,SKU001,12,12,0,0,38,ORDER,54,3.80,0.0,810.0
1,2026-10-01,SKU002,10,10,0,0,25,ORDER,158,5.00,0.0,4740.0
2,2026-10-01,SKU003,7,7,0,0,13,ORDER,101,1.95,0.0,2525.0
3,2026-10-02,SKU001,7,7,0,0,31,WAIT,0,3.10,0.0,0.0
4,2026-10-02,SKU002,13,13,0,0,12,WAIT,0,2.40,0.0,0.0
5,2026-10-02,SKU003,8,8,0,0,5,WAIT,0,0.75,0.0,0.0
6,2026-10-03,SKU001,10,10,0,0,21,WAIT,0,2.10,0.0,0.0
7,2026-10-03,SKU002,11,11,0,0,1,WAIT,0,0.20,0.0,0.0
8,2026-10-03,SKU003,9,5,4,0,0,WAIT,0,0.00,140.0,0.0
9,2026-10-04,SKU001,15,15,0,54,60,WAIT,0,6.00,0.0,0.0


Simulation completed successfully!


In [40]:

#PERFORMANCE EVALUATION

performance = simulation_results.groupby(
    "sku", as_index=False
).agg(
    total_demand=("demand", "sum"),
    units_fulfilled=("units_sold", "sum"),
    total_stockouts=("stockouts", "sum"),
    holding_cost=("holding_cost", "sum"),
    stockout_cost=("stockout_cost", "sum"),
    order_cost=("order_cost", "sum"),
    orders_placed=("order_quantity",
                   lambda x: (x > 0).sum())
)

# Calculate fill rate
performance["fill_rate"] = (
    performance["units_fulfilled"]
    / performance["total_demand"].replace(0, float("nan"))
    * 100
).fillna(100).round(2)

# Calculate total cost
performance["total_cost"] = (
    performance["holding_cost"]
    + performance["stockout_cost"]
    + performance["order_cost"]
)

performance.to_csv(
    "performance_evaluation.csv", index=False
)

print("MEMBER 5 – PERFORMANCE EVALUATION")
display(performance)

print("\nOVERALL PERFORMANCE")
print("Total Stockouts:",
      performance["total_stockouts"].sum())

total_demand = performance["total_demand"].sum()
total_fulfilled = performance["units_fulfilled"].sum()

overall_fill_rate = (
    total_fulfilled / total_demand * 100
    if total_demand > 0 else 100
)

print(f"Overall Fill Rate: {overall_fill_rate:.2f}%")
print(f"Total Holding Cost: ${performance['holding_cost'].sum():.2f}")
print(f"Total Stockout Cost: ${performance['stockout_cost'].sum():.2f}")
print(f"Total Order Cost: ${performance['order_cost'].sum():.2f}")
print(f"Total Cost: ${performance['total_cost'].sum():.2f}")


MEMBER 5 – PERFORMANCE EVALUATION


,sku,total_demand,units_fulfilled,total_stockouts,holding_cost,stockout_cost,order_cost,orders_placed,fill_rate,total_cost
0,SKU001,287,287,0,119.2,0.0,4935.0,6,100.00,5054.2
1,SKU002,459,429,30,362.8,1200.0,16830.0,5,93.46,18392.8
2,SKU003,243,203,40,164.1,1400.0,7425.0,5,83.54,8989.1



OVERALL PERFORMANCE
Total Stockouts: 70
Overall Fill Rate: 92.92%
Total Holding Cost: $646.10
Total Stockout Cost: $2600.00
Total Order Cost: $29190.00
Total Cost: $32436.10


### Member 6 – Agent Logs, Testing & Final Validation

This section validates the completed Inventory Replenishment Agent,
displays the agent's daily decisions, and tests the integrated system
to ensure that inventory, ordering, and performance outputs are valid.


In [41]:
import pandas as pd
import os

print("=" * 70)
print("DAILY AGENT DECISION LOG")
print("=" * 70)

# Load simulation results from the saved CSV file if not in memory
if 'simulation_results' not in locals() and 'simulation_results' not in globals():
    if os.path.exists("simulation_results.csv"):
        simulation_results = pd.read_csv("simulation_results.csv")
    else:
        simulation_results = pd.DataFrame()  # Empty fallback

log_columns = [
    "date",
    "sku",
    "demand",
    "ending_stock",
    "decision",
    "order_quantity"
]

# Include explanation/rationale if it was saved by the simulation
if not simulation_results.empty and "reason" in simulation_results.columns:
    log_columns.append("reason")

if not simulation_results.empty:
    print(simulation_results[log_columns].head(10))
else:
    print("Simulation results could not be loaded. Please run the simulation cell first.")

DAILY AGENT DECISION LOG
        date     sku  demand  ending_stock decision  order_quantity
0 2026-10-01  SKU001      12            38    ORDER              54
1 2026-10-01  SKU002      10            25    ORDER             158
2 2026-10-01  SKU003       7            13    ORDER             101
3 2026-10-02  SKU001       7            31     WAIT               0
4 2026-10-02  SKU002      13            12     WAIT               0
5 2026-10-02  SKU003       8             5     WAIT               0
6 2026-10-03  SKU001      10            21     WAIT               0
7 2026-10-03  SKU002      11             1     WAIT               0
8 2026-10-03  SKU003       9             0     WAIT               0
9 2026-10-04  SKU001      15            60     WAIT               0


In [42]:
# MEMBER 6: SYSTEM TESTING & VALIDATION
# ==========================================

print("=" * 70)
print("MEMBER 6 - SYSTEM TESTS")
print("=" * 70)

tests_passed = 0
tests_failed = 0


def run_test(test_name, condition):
    global tests_passed, tests_failed

    if condition:
        print(f"PASS: {test_name}")
        tests_passed += 1
    else:
        print(f"FAIL: {test_name}")
        tests_failed += 1
# TEST 1: Simulation produced results
run_test(
    "Simulation produced output",
    len(simulation_results) > 0
)


# TEST 2: Required columns exist
required_columns = [
    "date",
    "sku",
    "demand",
    "ending_stock",
    "decision",
    "order_quantity"
]

run_test(
    "All required simulation columns exist",
    all(col in simulation_results.columns
        for col in required_columns)
)


# TEST 3: Inventory never goes below zero
run_test(
    "Ending inventory is never negative",
    (simulation_results["ending_stock"] >= 0).all()
)


# TEST 4: Decisions are valid
run_test(
    "All decisions are ORDER or WAIT",
    simulation_results["decision"]
    .isin(["ORDER", "WAIT"])
    .all()
)


# TEST 5: WAIT decisions don't place orders
wait_rows = simulation_results[
    simulation_results["decision"] == "WAIT"
]

run_test(
    "WAIT decisions have zero order quantity",
    (wait_rows["order_quantity"] == 0).all()
)


# TEST 6: ORDER decisions have positive quantities
order_rows = simulation_results[
    simulation_results["decision"] == "ORDER"
]

run_test(
    "ORDER decisions have positive order quantities",
    len(order_rows) == 0 or
    (order_rows["order_quantity"] > 0).all()
)


# TEST 7: Demand values are valid
run_test(
    "Demand values are non-negative",
    (simulation_results["demand"] >= 0).all()
)


# FINAL TEST SUMMARY
print("\n" + "=" * 70)
print("TEST SUMMARY")
print("=" * 70)

print("Tests Passed:", tests_passed)
print("Tests Failed:", tests_failed)

if tests_failed == 0:
    print("\nALL MEMBER 6 TESTS PASSED SUCCESSFULLY!")
else:
    print("\nSome tests failed. Review the results above.")

MEMBER 6 - SYSTEM TESTS
PASS: Simulation produced output
PASS: All required simulation columns exist
PASS: Ending inventory is never negative
PASS: All decisions are ORDER or WAIT
PASS: WAIT decisions have zero order quantity
PASS: ORDER decisions have positive order quantities
PASS: Demand values are non-negative

TEST SUMMARY
Tests Passed: 7
Tests Failed: 0

ALL MEMBER 6 TESTS PASSED SUCCESSFULLY!


In [43]:
# MEMBER 6: FINAL SYSTEM SUMMARY
# ==========================================

print("=" * 70)
print("FINAL INVENTORY REPLENISHMENT AGENT SUMMARY")
print("=" * 70)

print(f"Total simulation records: {len(simulation_results)}")
print(f"SKUs tested: {simulation_results['sku'].nunique()}")

print(
    f"Total ORDER decisions: "
    f"{(simulation_results['decision'] == 'ORDER').sum()}"
)

print(
    f"Total WAIT decisions: "
    f"{(simulation_results['decision'] == 'WAIT').sum()}"
)

print(
    f"Total stockouts: "
    f"{simulation_results['stockouts'].sum()}"
)

if "holding_cost" in simulation_results.columns:
    print(
        f"Total holding cost: "
        f"${simulation_results['holding_cost'].sum():,.2f}"
    )

if "stockout_cost" in simulation_results.columns:
    print(
        f"Total stockout cost: "
        f"${simulation_results['stockout_cost'].sum():,.2f}"
    )

print("\nSystem validation complete.")

FINAL INVENTORY REPLENISHMENT AGENT SUMMARY
Total simulation records: 90
SKUs tested: 3
Total ORDER decisions: 16
Total WAIT decisions: 74
Total stockouts: 70
Total holding cost: $646.10
Total stockout cost: $2,600.00

System validation complete.


### Member 6 – Testing and Integration Summary

The completed Inventory Replenishment Agent was tested after integrating
the work from all group members. Daily agent decisions were reviewed to
confirm that the system produces valid ORDER and WAIT actions.

Validation tests checked that the simulation generated results, required
output fields were present, ending inventory remained non-negative,
ORDER decisions produced positive order quantities, WAIT decisions did
not generate orders, and demand values were valid.

The final outputs were also reviewed for stockouts, inventory levels,
ordering activity, and costs. These logs and tests improve transparency
by making the agent's behavior easier to review and verify.